# Lesson 01
Run lessons 1–0 first, or use 00_run_all.ipynb. This is the actual teaching source, not a placeholder.

# 第1课 研究基础 数据与趋势策略

**Quant Research and Execution Lab · 标准课件 01_foundations_data_and_trend.ipynb · 修订 2026-10-13**

目标：从一笔能手算的交易开始，用同一套现金与持仓账本研究 SPY 双均线、时间序列动量和买入持有。完成后应能解释每笔收益的时间来源、费用和现金，并说明趋势模型什么时候失败。按下方01至09顺序运行，所有代码直接展示在本笔记本中。

**金融直觉与术语**

ETF是可在交易所买卖的一篮子资产基金；SPY是本课预设研究对象。做多是在价格上涨时获益；做空是先卖借来的资产；杠杆是超过自有本金的敞口。本课只做多或持有现金，不借券、不融资。基准是同起点的买入持有，帮助判断规则是否优于简单持有。回撤是账户权益相对此前峰值的跌幅，不等于某一天的跌幅。

趋势假设：价格变化可能延续。双均线用短期平均价格高于长期平均价格作为持有条件；时间序列动量用本资产过去126个交易日收益为正作为持有条件。它们不是上涨保证，震荡时可能反复买卖。20/100、50/200、126是预先登记的教学窗口，不是已经找到的最优参数。

**符号与公式**

$C_t$为调整后的收盘价，$O_t$为同口径开盘价，$s_t$是收盘信号，$w_t$为开盘目标权重，$q_t$为模型持仓单位，$B_t$为现金，$E_t$为账户权益，$c$为单边成本率。

$$SMA_{n,t}=\frac{1}{n}\sum_{j=0}^{n-1}C_{t-j},\qquad s_t^{MA}=1(SMA_{20,t}>SMA_{100,t})$$
$$MOM_{L,t}=C_t/C_{t-L}-1,\qquad s_t^{MOM}=1(MOM_{126,t}>0),\qquad w_t=s_{t-1}$$

周一收盘形成信号 → 周二开盘成交并付费 → 周三开盘估值。策略只能获得买入之后的收益。信号用过去和当日数据；下一开盘只能用于事后评估，不能进入当日决策。252个交易日统一预热，所有模型同一起点，预热时不交易。

$$E_{pre}=B+qO_t,\quad F=|\Delta q|O_t c,\quad B'=B-\Delta q O_t-F$$
$$E_{end}=B'+(q+\Delta q)O_{t+1},\quad r_t=E_{end}/E_{pre}-1$$
$$V_t=\prod_{i\leq t}(1+r_i),\quad DD_t=V_t/\max(1,V_1,\ldots,V_t)-1$$

本课用现金约束精确求解扣费后的目标持仓，不能先把全部现金买完，再另扣费用导致透支。年化收益为$V_T^{252/N}-1$；年化波动为日收益标准差乘$\sqrt{252}$；零无风险率夏普为日均收益除以日标准差乘$\sqrt{252}$。252是日频股票的近似，不能套到逐笔数据。

**手算示例与参考答案**

本金1000元，100元买5份，成交额500元。5bp即0.05%，费用0.25元，剩余现金499.75元。下一开盘102元时，权益为499.75+5×102=1009.75元，账户收益0.975%；若98元则权益989.75元，收益−1.025%。标的涨2%不代表整个账户涨2%，因为部分本金仍是现金。

**数据口径与边界**

默认只下载2008—2022年开发数据，end='2023-01-01'是排他的结束日期。价格索引来自真实交易日，节假日不补虚构价格；缺失值先停止核对，不使用未来值回填。保存vendor Open/Close、Adj Close、调整因子和分红拆股事件。Yahoo的非股息复权字段也不能直接视为交易所原始逐笔价格。

账户算法使用调整后的价格和**模型单位**，便于统一总回报研究。它们不是历史真实报价和可下单股数；复权已包含公司行动的调整，账本不再次发分红或增加拆股股数。该简化不能验证真实支付日、税、整数股、成交容量和冲击；进入实盘账本前必须另建原始价格及公司行动处理。调整序列可能被供应商修订，本次哈希用于标识版本，不保证未来重新下载字节完全相同。

2023—2026年9月原规划为历史留出；此前笔记本已显示后期表现，故不能再宣称完全未见的测试。此课不下载、不据其调参。后续冻结规则后须如实记录已接触数据，使用真正未用的数据或前瞻模拟检验。SPY固定资产研究不等于已处理全市场幸存者偏差。

**代码阅读路线**

|单元格|运行产出与逐段解释|
|---|---|
|01 环境配置|固定依赖、种子、research_config；字典记录研究设置，CPU足够。|
|02 手算|核对现金和权益；rolling窗口形成后才有均线；shift延迟一行即下一交易日。|
|03 数据|下载或显式CSV接口，检查字段与重复，保留事件、覆盖、版本、SHA256与失败日志。|
|04 信号|金融规则转成0/1目标；instrument、model_id、available_at、horizon统一供后续模块使用。|
|05 账本|simulate是共用函数；成交额、费用、现金、持仓、权益、换手与暴露逐日记录。|
|06 验证与基线|假设数据验证记账；改未来价格不改变过去信号；三个模型同起点同费用。|
|07 指标|看收益、夏普、最大回撤、订单数、换手和暴露；分段只是开发稳定性观察。|
|08 三张图|价格与均线、净/毛权益和基准、回撤与暴露；净值点画在收益结束时点。|
|09 对照实验|5/10/20bp成本、50/200慢均线、额外延迟一日；每次只改变一个因素，保留负结果。|

**调优问题与练习参考**

1. 基础实验：顺序运行全部单元格，手算核对02和06，解释signal、target_weight和执行时点。参考：周一收盘信息在周一开盘不可得，shift(1)将指令推至周二。
2. 成本实验：比较5/10/20bp。参考：它们是教学总成本情景，不是实际券商费率；相同0/1指令成本提高，终值应不升，具体差额读取09的实际输出。
3. 个人对照：从慢均线或额外延迟中选一项，先写假设，再报告实际收益、回撤、订单数和市场暴露。参考：慢均线可能减少反应，也可能错过趋势；延迟可好可坏，不预先写结论。
4. 机制讨论：策略赚的是趋势择时还是长期市场暴露？参考：结合买入持有、持仓比例和分段结果，单凭高夏普不能证明独立超额收益。
5. 进阶：预先登记一个过去窗口最高价突破信号，复用make_interface和simulate，在相同预热和开发区间比较。不要在最终留出期选择窗口，不必以复杂模型作为通过标准。

**失败条件与验收**

Yahoo失败、字段缺失、重复日期、价格非正、样本不足立即停止；不静默替换成模拟真实行情。频繁震荡、趋势逆转、跳空、滑点上升、市场暴露降低和供应商修订都可能使结论失效。数据哈希、版本和试验记录保留在输出；当前课不向本地写CSV、图片或额外课件文件。

验收：在新Colab会话自上而下运行；费用方向、现金约束、记账守恒、因果性与共同窗口检查通过；图表有结果；能解释复权局限并独立完成一项对照。Notebook保存不等于运行内存永久保存，依赖输出和数据记录需随课件保留。个人作业尚需学习者完成，不能由代码跑通替代。

**官方资料**

- [Colab FAQ：会话、保存与资源](https://research.google.com/colaboratory/faq.html)
- [yfinance download：日期、调整与事件参数](https://ranaroussi.github.io/yfinance/reference/api/yfinance.download.html)
- [yfinance项目与数据使用说明](https://github.com/ranaroussi/yfinance)
- [pandas rolling](https://pandas.pydata.org/docs/reference/api/pandas.Series.rolling.html) · [shift](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.shift.html)
- [Backtesting.py：成交、佣金与时点参考](https://kernc.github.io/backtesting.py/doc/backtesting/backtesting.html)

---
下面所有单元格保持展开。先全部运行标准版，再逐段学习；从本课开始复用相同信号接口和账本。


In [ ]:
# 01 环境与研究配置：CPU 足够；依赖安装只发生在 Colab
%pip -q install numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.3 yfinance==0.2.66
import sys, json, hashlib, platform
from datetime import datetime, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from IPython.display import display, Markdown

SEED = 42
np.random.seed(SEED)
research_config = {
    "lesson": "01_foundations_data_and_trend",
    "instrument": "SPY", "start": "2008-01-01", "end_exclusive": "2023-01-01",
    "fast": 20, "slow": 100, "momentum_lookback": 126,
    "common_warmup": 252, "initial_cash": 10000.0,
    "one_way_cost_bps": 5.0, "execution_delay_sessions": 1,
    "cash_rate": 0.0, "risk_free_rate": 0.0, "annual_sessions": 252,
    "accounting": "adjusted_price_synthetic_units", "seed": SEED,
    "development": "2008-2022", "historical_holdout": "2023-2026-09",
    "holdout_status": "previous notebook displayed later results; not pristine unseen"
}
# 默认不下载、不查看历史留出区间，不以它选择参数。
run_log = {"started_utc": datetime.now(timezone.utc).isoformat(),
           "python": sys.version, "platform": platform.platform(),
           "versions": {p: version(p) for p in ["numpy", "pandas", "matplotlib", "yfinance"]},
           "seed": SEED, "status": "started"}
experiment_registry = []
print(json.dumps(research_config, ensure_ascii=False, indent=2))
print("实际依赖版本：", run_log["versions"])
# 字典存研究设置；函数复用同一算法；DataFrame 按交易日索引存价格和账本。


In [ ]:
# 02 手算账本：假设数据，与 Yahoo 行情明确分开
initial_cash = 1000.0
quantity, entry_price, next_price = 5.0, 100.0, 102.0
cost_rate = 5 / 10000  # 1 bp = 0.01%；5 bp = 0.05%
notional = quantity * entry_price
fee = notional * cost_rate
cash = initial_cash - notional - fee
equity = cash + quantity * next_price
print(f"成交额={notional:.2f}，费用={fee:.2f}，剩余现金={cash:.2f}")
print(f"下一开盘权益={equity:.2f}，账户收益={equity/initial_cash-1:.3%}")
assert np.isclose(equity, 1009.75)
# 参考答案：跌至98元，权益989.75元，收益-1.025%。
print("跌至98元：", cash + quantity*98, (cash+quantity*98)/initial_cash-1)
# 手算 rolling 和 shift：3日均线在第3个交易日才形成。
toy = pd.DataFrame({"close": [100., 102., 104., 99., 98.]},
                   index=pd.date_range("2020-01-06", periods=5, freq="B"))
toy["sma3"] = toy.close.rolling(3, min_periods=3).mean()
toy["signal_at_close"] = (toy.close > toy.sma3).astype(int)
toy["target_at_next_open"] = toy.signal_at_close.shift(1).fillna(0)
display(toy)
# 第3天均线=(100+102+104)/3=102；信号1在第4天开盘才能执行。
assert toy.iloc[2].sma3 == 102 and toy.iloc[3].target_at_next_open == 1


In [ ]:
# 03 数据接口、原始/调整口径、事件与数据清单
# Yahoo字段的“未做股息调整”不等于交易所原始逐笔价；历史价格仍可能已按拆股调整。
SOURCE = "yahoo"  # 下载失败可改"csv"，指定Colab中已有CSV；不自动伪造真实数据。
CSV_PATH = None  # CSV需Date, Open, Close, Adj Close；事件列可选。
def load_prices(source="yahoo", csv_path=None):
    if source == "yahoo":
        data = yf.download(research_config["instrument"],
            start=research_config["start"], end=research_config["end_exclusive"],
            auto_adjust=False, actions=True, multi_level_index=False,
            progress=False, threads=False)
    elif source == "csv" and csv_path:
        data = pd.read_csv(csv_path, index_col="Date", parse_dates=True)
        data = data.loc[(data.index >= research_config["start"]) &
                        (data.index < research_config["end_exclusive"])]
    else:
        raise ValueError("请使用yahoo或提供Colab中CSV路径")
    if data is None or data.empty:
        raise RuntimeError("行情下载失败：检查Yahoo网络/限流。停止研究，不填入合成行情。")
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)
    data = data.sort_index().copy()
    data.index = pd.DatetimeIndex(data.index).tz_localize(None).normalize()
    if data.index.has_duplicates:
        raise ValueError("发现重复交易日，需核对来源")
    needed = ["Open", "Close", "Adj Close"]
    if not set(needed).issubset(data.columns):
        raise ValueError("缺少Open/Close/Adj Close，不能静默混用价格口径")
    data[needed] = data[needed].apply(pd.to_numeric, errors="coerce")
    if data[needed].isna().any().any() or (data[needed] <= 0).any().any():
        raise ValueError("价格缺失或非正数：先核对，不用未来值回填")
    if len(data) < 500:
        raise ValueError("样本不足，先核对覆盖范围")
    return data
try:
    raw = load_prices(SOURCE, CSV_PATH)
except Exception as exc:
    run_log.update(status="data_failed", failure=str(exc))
    print(json.dumps(run_log, ensure_ascii=False, indent=2))
    raise
factor = raw["Adj Close"] / raw["Close"]
prices = pd.DataFrame({"vendor_open": raw.Open, "vendor_close": raw.Close,
    "adjustment_factor": factor, "open": raw.Open*factor, "close": raw["Adj Close"]})
for name in ["Dividends", "Stock Splits"]:
    prices[name] = raw[name].fillna(0) if name in raw else 0.0
prices.index.name = "session"
prices["close_available_at"] = (prices.index + pd.Timedelta(hours=16)).tz_localize("America/New_York")
prices["open_execution_at"] = (prices.index + pd.Timedelta(hours=9, minutes=30)).tz_localize("America/New_York")
# CSV字节仅在内存生成，用哈希识别本次数据；不在本地落盘。
data_manifest = {"source": SOURCE, "instrument": research_config["instrument"],
    "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    "rows": len(raw), "first_session": str(raw.index.min().date()),
    "last_session": str(raw.index.max().date()),
    "sha256": hashlib.sha256(raw.to_csv(float_format="%.10g").encode()).hexdigest(),
    "price_basis": "Yahoo adjusted OHLC synthetic units; vendor fields retained",
    "events_present": [x for x in ["Dividends", "Stock Splits"] if x in raw],
    "license": "教学个人研究；发布前另核验Yahoo条款，不默认再分发行情"}
print(json.dumps(data_manifest, ensure_ascii=False, indent=2))
display(prices.head(), prices.tail())
display(prices.loc[(prices["Dividends"] != 0) | (prices["Stock Splits"] != 0)].tail())
# 交易日只取数据索引：节假日不造价格；星期一的上一行通常是星期五。
# 复权已反映公司行动，本课模型账本不再次加分红或乘拆股，避免重复记账。


In [ ]:
# 04 两类趋势信号与统一时间接口
# SMA_n(t)=过去n个收盘价均值；MOM_L(t)=C_t/C_(t-L)-1。
# 两种规则都是做多/现金：不卖空、不加杠杆。20/100和126是预设教学窗口。
def make_signal(close, model="ma", fast=20, slow=100, lookback=126):
    if model == "ma":
        if not 1 < fast < slow: raise ValueError("需要1<fast<slow")
        a = close.rolling(fast, min_periods=fast).mean()
        b = close.rolling(slow, min_periods=slow).mean()
        return (a > b).where(b.notna()).astype(float)
    if model == "mom":
        m = close/close.shift(lookback)-1
        return (m > 0).where(m.notna()).astype(float)
    if model == "buy_hold": return pd.Series(1.0, index=close.index)
    raise ValueError("未知模型")

def make_interface(model="ma", fast=20, slow=100, lookback=126, delay=1):
    if delay < 1: raise ValueError("至少延迟1个交易日，禁止收盘信号当天开盘执行")
    s = make_signal(prices.close, model, fast, slow, lookback)
    out = pd.DataFrame({"instrument": research_config["instrument"],
        "model_id": f"{model}_{fast}_{slow}_{lookback}_delay{delay}",
        "signal": s, "available_at": prices.close_available_at,
        "target_weight": s.shift(delay),
        "decision_available_at": prices.close_available_at.shift(delay),
        "execution_at": prices.open_execution_at, "horizon": "next_open"})
    # 252个预热交易日供所有参数共用；预热不交易；最后一行只做下一开盘估值。
    out = out.iloc[research_config["common_warmup"]:-1].copy()
    if out.target_weight.isna().any(): raise ValueError("预热不足")
    assert (out.decision_available_at < out.execution_at).all()
    return out

signal_table = make_interface()
features = prices[["close"]].copy()
features["sma20"] = prices.close.rolling(20).mean()
features["sma100"] = prices.close.rolling(100).mean()
features["momentum126"] = prices.close/prices.close.shift(126)-1
display(signal_table.head(), features.tail())
# rolling不指定center=True；shift(+1)延后指令；shift(-1)仅用于事后评估下一开盘。
# available_at使用纽约时间16:00保守记录；早收盘日仍晚于真实收盘但早于下一开盘。


In [ ]:
# 05 共用交易账本：按成交额计费，买入费用由现金支付，不能透支
# 本课使用复权价格和“模型单位”，不是实际券商股数/成交报价。
# E_pre=cash+q*O；F=|delta_q|*O*c；cash'=cash-delta_q*O-F。
# E_end=cash'+q'*O_next；目标权重按扣费后权益求解，避免先满仓再产生负现金。
def simulate(open_prices, target_weight, cost_bps=5.0, initial=10000.0):
    if not 0 <= cost_bps < 10000 or initial <= 0: raise ValueError("成本或本金无效")
    p = open_prices.reindex(target_weight.index)
    next_p = open_prices.shift(-1).reindex(target_weight.index)
    next_date = pd.Series(open_prices.index, index=open_prices.index).shift(-1).reindex(target_weight.index)
    if p.isna().any() or next_p.isna().any(): raise ValueError("缺少执行或下一开盘价格")
    if (p <= 0).any() or (next_p <= 0).any(): raise ValueError("价格必须为正")
    if not target_weight.between(0,1).all(): raise ValueError("只允许0到1权重")
    cash, units, c, rows = float(initial), 0.0, cost_bps/10000, []
    for date, weight in target_weight.items():
        price, next_price = float(p.loc[date]), float(next_p.loc[date])
        before = cash + units*price
        # 求解d = w*(E_pre-c*abs(d))-旧持仓市值；d为有方向成交额。
        gap = float(weight)*before - units*price
        signed_notional = gap/(1+float(weight)*c*np.sign(gap))
        if abs(signed_notional) < 1e-8: signed_notional = 0.0
        delta_units = signed_notional/price
        fee = abs(signed_notional)*c
        cash = cash-signed_notional-fee
        units = units+delta_units
        after = cash+units*price
        end = cash+units*next_price
        assert np.isclose(after, before-fee, rtol=1e-10, atol=1e-7)
        assert cash >= -1e-7 and units >= -1e-7
        rows.append({"session": date, "end_session": next_date.loc[date],
            "target_weight": weight, "model_open": price, "next_model_open": next_price,
            "equity_before": before, "delta_model_units": delta_units,
            "trade_notional": abs(signed_notional), "fee": fee, "cash": cash,
            "model_units": units, "equity_after_trade": after, "equity_end": end,
            "net_return": end/before-1, "turnover": abs(signed_notional)/before,
            "exposure": units*price/after if after > 0 else np.nan})
    ledger = pd.DataFrame(rows).set_index("session")
    ledger["wealth"] = ledger.equity_end/initial
    ledger["drawdown"] = ledger.wealth/ledger.wealth.cummax().clip(lower=1)-1
    return ledger
# 后续课程使用同一simulate接口，不为每个信号重写收益算法。
# end_session表示收益终点；最后一个执行日之后仍有一个开盘估值日。
# 不强制末日卖出；现金利息、税、整数股约束和实际冲击本课暂未建模。


In [ ]:
# 06 关键验证与标准实验：先验证算法，再解释真实数据结果
# 用标注的假设价格验证费用方向、账本守恒和未来信息边界。
test_open = pd.Series([100., 102., 101.], index=pd.date_range("2020-01-06", periods=3, freq="B"))
test_weight = pd.Series([1., 0.], index=test_open.index[:2])
test_ledger = simulate(test_open, test_weight, 5, 1000)
expected_units = 1000/(100*1.0005)
expected_final = expected_units*102*(1-.0005)
assert np.isclose(test_ledger.iloc[-1].equity_end, expected_final)
assert np.isclose(test_ledger.fee.sum(), expected_units*(100+102)*.0005)
free = simulate(test_open, test_weight, 0, 1000)
assert test_ledger.iloc[-1].equity_end <= free.iloc[-1].equity_end
all_cash = simulate(test_open, test_weight*0, 20, 1000)
assert np.allclose(all_cash.equity_end,1000) and all_cash.fee.sum()==0
assert np.allclose(test_ledger.equity_end, test_ledger.cash+test_ledger.model_units*test_ledger.next_model_open)
assert np.allclose((1+test_ledger.net_return).cumprod(),test_ledger.wealth)
# 改写后半段的未来收盘价，过去信号必须完全不变。
cut = len(prices)//2
changed = prices.close.copy()
changed.iloc[cut:] *= 7
for model in ["ma", "mom"]:
    pd.testing.assert_series_equal(make_signal(prices.close,model).iloc[:cut],
                                   make_signal(changed,model).iloc[:cut])
print("PASS：买卖费用、无透支、现金策略、记账守恒、复利及信号因果性")

interfaces, ledgers = {}, {}
for name, model in [("SMA20/100","ma"),("Momentum126","mom"),("BuyHold","buy_hold")]:
    interfaces[name] = make_interface(model)
    ledgers[name] = simulate(prices.open, interfaces[name].target_weight,
                            research_config["one_way_cost_bps"],research_config["initial_cash"])
ledgers["SMA gross"] = simulate(prices.open,interfaces["SMA20/100"].target_weight,0,
                                research_config["initial_cash"])
common_index = ledgers["SMA20/100"].index
assert all(x.index.equals(common_index) for x in ledgers.values())
experiment_registry.append({"id":"baseline", "data_sha256":data_manifest["sha256"],
    "config":research_config.copy(), "models":list(ledgers), "scope":"development only"})
ledger = ledgers["SMA20/100"].join(interfaces["SMA20/100"][["decision_available_at","execution_at"]])
ledger["instrument"] = research_config["instrument"]
ledger["model_id"] = "SMA20/100"
display(ledger.head())
display(ledger.loc[ledger.trade_notional>1e-8].tail(10))
print("所有模型同起点、同终点、同本金、同成本口径；BuyHold初始买入也扣费。")


In [ ]:
# 07 收益、风险与暴露：都从共同账本计算
# r_t=E_end/E_pre-1；净值=prod(1+r)；回撤=净值/历史峰值-1。
# 年化收益用几何复利；夏普用日均收益/日标准差*sqrt(252)，本课Rf=0。
def performance(frame):
    r = frame.net_return.dropna()
    if len(r) < 2: return {}
    wealth = (1+r).cumprod()
    std = r.std(ddof=1)
    return {"total_return": wealth.iloc[-1]-1,
        "annual_return": wealth.iloc[-1]**(252/len(r))-1,
        "annual_vol": std*np.sqrt(252),
        "sharpe_rf0": r.mean()/std*np.sqrt(252) if std>1e-12 else np.nan,
        "max_drawdown": (wealth/wealth.cummax().clip(lower=1)-1).min(),
        "order_count": int((frame.trade_notional>1e-8).sum()),
        "turnover_sum": frame.turnover.sum(), "mean_exposure": frame.exposure.mean(),
        "fee_cash_sum": frame.fee.sum()}
summary = pd.DataFrame({name:performance(x) for name,x in ledgers.items()}).T
percentage_cols = ["total_return","annual_return","annual_vol","max_drawdown","mean_exposure"]
formats = {x:"{:.2%}" for x in percentage_cols}
formats.update({"sharpe_rf0":"{:.2f}","turnover_sum":"{:.2f}",
                "order_count":"{:.0f}","fee_cash_sum":"{:.2f}"})
print("开发区间执行起点：",common_index[0].date(),"最后估值：",ledger.end_session.iloc[-1].date())
display(summary.style.format(formats))
# 分段仅作开发区间稳定性观察，每段按该段收益重新归一，不声称独立测试。
period_results=[]
for label, start, end in [("2009-2012","2009-01-01","2013-01-01"),
                          ("2013-2017","2013-01-01","2018-01-01"),
                          ("2018-2022","2018-01-01","2023-01-01")]:
    for name,x in ledgers.items():
        sub=x.loc[(x.index>=start)&(x.index<end)]
        period_results.append({"period":label,"model":name,**performance(sub)})
display(pd.DataFrame(period_results).set_index(["period","model"]).style.format(formats))
# 费用金额合计不能直接从累计收益减去；费用改变以后可投资本金和复利路径。
# 低波动可能只是现金持有更多；结合mean_exposure判断，不能只看夏普。


In [ ]:
# 08 三张图：价格信号、同窗口净值、风险与市场暴露
fig,ax=plt.subplots(figsize=(13,4))
for col,label in [("close","Adjusted close"),("sma20","SMA20"),("sma100","SMA100")]:
    ax.plot(features.loc[common_index,col],label=label,lw=1)
ax.set(title="1. SPY adjusted-price trend signals (development)",ylabel="Adjusted model price")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

fig,ax=plt.subplots(figsize=(13,4))
for name,x in ledgers.items():
    # 净值点放在收益终点，并显式补初始权益1；不把下一开盘结果画到当天开盘。
    dates=pd.DatetimeIndex([x.index[0],*x.end_session.tolist()])
    ax.plot(dates,np.r_[1.,x.wealth.to_numpy()],label=name,lw=1.5)
ax.set(title="2. Wealth: identical starting capital, dates and accounting",ylabel="Initial wealth = 1")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

fig,axes=plt.subplots(2,1,figsize=(13,6),sharex=True)
for name in ["SMA20/100","Momentum126","BuyHold"]:
    x=ledgers[name]
    axes[0].plot(x.end_session,100*x.drawdown,label=name)
    axes[1].plot(x.index,x.exposure,label=name,alpha=.7)
axes[0].set(title="3. Drawdown and exposure",ylabel="Drawdown (%)")
axes[1].set(ylabel="Equity in market",xlabel="Trading session",ylim=(-.05,1.05))
for ax in axes: ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()
# 读图：净值较高是否伴随更深回撤？策略持有现金时是否错过上涨？
# 第一张为研究复权价；第二张需同时看净/毛；第三张把风险和现金暴露联系起来。


In [ ]:
# 09 个人对照实验、参考解答和运行记录
# 一次只改变一个因素，不搜索“最佳”参数，不读取历史留出期。
scenarios=[("baseline",20,100,5,1),("cost10",20,100,10,1),
           ("cost20",20,100,20,1),("slower50/200",50,200,5,1),
           ("delay2",20,100,5,2)]
scenario_ledgers, experiment_rows = {}, []
for name, fast, slow, bps, delay in scenarios:
    interface=make_interface("ma",fast=fast,slow=slow,delay=delay)
    result=simulate(prices.open,interface.target_weight,bps,research_config["initial_cash"])
    assert result.index.equals(common_index)
    scenario_ledgers[name]=result
    experiment_rows.append({"experiment":name,"fast":fast,"slow":slow,"cost_bps":bps,
        "delay_sessions":delay,**performance(result)})
    experiment_registry.append({"id":name,"model":"ma","fast":fast,"slow":slow,
        "cost_bps":bps,"delay":delay,"data_sha256":data_manifest["sha256"],
        "scope":"development only; predeclared comparison"})
experiment_table=pd.DataFrame(experiment_rows).set_index("experiment")
display(experiment_table.style.format(formats))
assert (scenario_ledgers["baseline"].equity_end.iloc[-1] >=
        scenario_ledgers["cost10"].equity_end.iloc[-1] >=
        scenario_ledgers["cost20"].equity_end.iloc[-1])
fig,ax=plt.subplots(figsize=(13,4))
for name,x in scenario_ledgers.items(): ax.plot(x.end_session,x.wealth,label=name)
ax.set(title="One-factor-at-a-time experiments: development only",ylabel="Wealth")
ax.legend(); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

base=experiment_table.loc["baseline"]
print("自动参考解答（数值由本次数据计算）：")
print(f"基线年化收益{base.annual_return:.2%}，夏普{base.sharpe_rf0:.2f}，最大回撤{base.max_drawdown:.2%}")
for name in ["cost10","cost20","slower50/200","delay2"]:
    x=experiment_table.loc[name]
    print(f"{name}: 累计收益相对基线变化{x.total_return-base.total_return:+.2%}，"
          f"订单数{x.order_count:.0f}，夏普{x.sharpe_rf0:.2f}，回撤{x.max_drawdown:.2%}")
print("费用提高时，相同0/1信号的终值应不升；慢均线/额外延迟的表现不能预先断言。")
print("换仓次数是买入与卖出订单的总数，不等于完整往返交易次数。")
print("个人作业：选一个对照，先写假设，再解释表中实际结果、金融机制和失败证据。")
run_log.update(status="completed_in_current_session", finished_utc=datetime.now(timezone.utc).isoformat(),
    checks="accounting, fees, no_negative_cash, causality, timing, common_window, cost_direction passed",
    data_sha256=data_manifest["sha256"], experiment_count=len(experiment_registry),
    clean_session_note="重启运行时后全部运行；以实际运行结果核验，不据代码存在声称通过")
print("数据清单、配置、版本、实验记录均保留在本笔记本输出和内存；不自动写本地文件。")
print(json.dumps(run_log,ensure_ascii=False,indent=2))
display(pd.DataFrame(experiment_registry))
# 研究结论只限本数据版本与开发区间；负结果同样保留，不用复杂模型掩盖失败。
